# Format versus correctness comparison v2

Five arms, five seeds, 16 Adam steps. The confirm split decides. The 64-task pack is a secondary curve. Re-running the last cell continues `progress.jsonl`. `--check` does not train.

Use a GPU runtime. The commands match `docs/format-vs-correctness-comparison-v2.md`.

In [ ]:
import os

os.environ.setdefault("VARE_REPO", "https://github.com/mitukx/VARE.git")
os.environ.setdefault("VARE_REV", "cursor/format-vs-correctness-v2-af75")
print(os.environ["VARE_REPO"], os.environ["VARE_REV"])


In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/vare-format-vs-correctness-v2")
if not (repo / ".git").exists():
    subprocess.check_call(["git", "clone", os.environ["VARE_REPO"], str(repo)])
subprocess.check_call(["git", "fetch", "origin", os.environ["VARE_REV"]], cwd=repo)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        "/content/vare-format-vs-correctness-v2",
        "transformers",
        "accelerate",
        "peft",
        "huggingface_hub",
    ]
)


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/vare")
root.mkdir(parents=True, exist_ok=True)
output = root / "format-vs-correctness-comparison-v2-transformers"
output.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(root / "huggingface-cache")
os.environ["HF_HUB_CACHE"] = str(root / "huggingface-cache" / "hub")
print("output", output)


In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-format-vs-correctness-v2")
expected = {
    "protocols/format_vs_correctness_comparison_v2.json": "8214a9c784035f47b20d9e44fc6bbbd3cef45f821d86d6eb2f69936cc25eb6b4",
    "results/heldout-exact-integer-update-cost-smoke-v1/m1-mlx-f22b3b9/manifest.json": "8926c6c6383201bb3f1a067707210db7abd94c90a10f40ae0686784bc4ce511e",
    "benchmarks/heldout/exact_integer_v1/tasks.jsonl": "189b5e482b0f8c2b40ff36e0e1a84827699f348c11981a0ff912ce67323fd7fc",
    "benchmarks/heldout/integer_seed_split_v1/splits.json": "4544a9a852f615f7cf9c52d7a4826ed70ea6c5f57829754ac2e10f3c1fea4287",
}
for name, digest in expected.items():
    got = hashlib.sha256((repo / name).read_bytes()).hexdigest()
    print(name, got)
    assert got == digest
checked = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_format_vs_correctness_comparison_v2.py"),
        "--backend",
        "transformers",
        "--output",
        str(output),
        "--check",
    ],
    cwd=repo,
)
print("check_exit", checked.returncode)
completed = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_format_vs_correctness_comparison_v2.py"),
        "--backend",
        "transformers",
        "--output",
        str(output),
    ],
    cwd=repo,
)
print("comparison_exit", completed.returncode)
